# Generate Datasets

This notebook generates the reproducible datasets used throughout the Parameter Learning chapter and later robotics-oriented chapters.

A fixed random seed is used for every dataset so that all results are reproducible.


## Dataset Structure

```text
datasets/
├── discrete/
│   ├── coin_toss.csv
│   └── dice_rolls.csv
├── continuous/
│   └── gaussian_samples.csv
└── robotics/
    ├── robot_sensor_log.csv
    └── robot_sensor_missing.csv
```


In [11]:
from pathlib import Path
import numpy as np
import pandas as pd

DATASET_ROOT = Path("../Datasets/DecisionMakingDatasets")
for folder in [
    DATASET_ROOT / "discrete",
    DATASET_ROOT / "continuous",
    DATASET_ROOT / "robotics",
]:
    folder.mkdir(parents=True, exist_ok=True)


## 1. Coin Toss Dataset

True probabilities:

$$
P(H)=0.65,\qquad P(T)=0.35.
$$

This will be used for Bernoulli maximum-likelihood and Bayesian parameter learning.


In [12]:
rng = np.random.default_rng(42)

coin_tosses = rng.choice(
    ["H", "T"],
    size=1000,
    p=[0.65, 0.35],
)

coin_df = pd.DataFrame({
    "trial": np.arange(1, 1001),
    "outcome": coin_tosses,
})

coin_df.to_csv(
    DATASET_ROOT / "discrete" / "coin_toss.csv",
    index=False,
)

coin_df.head()


,trial,outcome
0,1,T
1,2,H
2,3,T
3,4,T
4,5,H


## 2. Dice Roll Dataset

The die is intentionally non-uniform, with true probabilities

\[
[0.10,\,0.15,\,0.25,\,0.20,\,0.18,\,0.12].
\]


In [4]:
rng = np.random.default_rng(43)

dice_probabilities = np.array([
    0.10, 0.15, 0.25,
    0.20, 0.18, 0.12,
])

dice_rolls = rng.choice(
    np.arange(1, 7),
    size=500,
    p=dice_probabilities,
)

dice_df = pd.DataFrame({
    "trial": np.arange(1, 501),
    "roll": dice_rolls,
})

dice_df.to_csv(
    DATASET_ROOT / "discrete" / "dice_rolls.csv",
    index=False,
)

dice_df.head()


,trial,roll
0,1,4
1,2,1
2,3,1
3,4,5
4,5,4


## 3. Multivariate Gaussian Dataset

Generated from

$$
\mu=\begin{bmatrix}
5.0\\
2.5
\end{bmatrix},
\qquad
\Sigma=\begin{bmatrix}
1.44 & 0.54\\
0.54 & 0.81
\end{bmatrix}.
$$


In [5]:
rng = np.random.default_rng(44)

true_mean = np.array([5.0, 2.5])

true_covariance = np.array([
    [1.44, 0.54],
    [0.54, 0.81],
])

samples = rng.multivariate_normal(
    mean=true_mean,
    cov=true_covariance,
    size=1000,
)

gaussian_df = pd.DataFrame({
    "x1": samples[:, 0],
    "x2": samples[:, 1],
})

gaussian_df.to_csv(
    DATASET_ROOT / "continuous" / "gaussian_samples.csv",
    index=False,
)

gaussian_df.head()


,x1,x2
0,3.305205,1.609971
1,4.224980,2.981426
2,3.850584,2.311841
3,5.793746,2.125202
4,4.748743,1.938277


## 4. Robot Sensor Dataset

A simulated robot follows a smooth 2D trajectory. We store ground truth together with noisy GPS, LiDAR, velocity, and heading measurements.

Because the injected sensor noise is known, later notebooks can compare learned parameters with ground truth.


In [6]:
rng = np.random.default_rng(45)

n_steps = 400
dt = 0.1
time = np.arange(n_steps) * dt

true_velocity = (
    1.2
    + 0.25 * np.sin(0.18 * time)
    + 0.08 * np.cos(0.05 * time)
)

true_heading = (
    0.15 * np.sin(0.12 * time)
    + 0.04 * np.cos(0.03 * time)
)

true_x = np.zeros(n_steps)
true_y = np.zeros(n_steps)

for i in range(1, n_steps):
    true_x[i] = (
        true_x[i - 1]
        + true_velocity[i - 1]
        * np.cos(true_heading[i - 1])
        * dt
    )
    true_y[i] = (
        true_y[i - 1]
        + true_velocity[i - 1]
        * np.sin(true_heading[i - 1])
        * dt
    )


In [7]:
gps_noise = rng.multivariate_normal(
    [0.0, 0.0],
    [[0.36, 0.08], [0.08, 0.25]],
    size=n_steps,
)

lidar_noise = rng.multivariate_normal(
    [0.0, 0.0],
    [[0.09, 0.02], [0.02, 0.06]],
    size=n_steps,
)

velocity_noise = rng.normal(
    0.0, 0.08, size=n_steps
)

heading_noise = rng.normal(
    0.0, 0.025, size=n_steps
)

robot_df = pd.DataFrame({
    "time_s": time,
    "true_x_m": true_x,
    "true_y_m": true_y,
    "true_velocity_mps": true_velocity,
    "true_heading_rad": true_heading,
    "gps_x_m": true_x + gps_noise[:, 0],
    "gps_y_m": true_y + gps_noise[:, 1],
    "lidar_x_m": true_x + lidar_noise[:, 0],
    "lidar_y_m": true_y + lidar_noise[:, 1],
    "velocity_measurement_mps": true_velocity + velocity_noise,
    "heading_measurement_rad": true_heading + heading_noise,
    "gps_error_x_m": gps_noise[:, 0],
    "gps_error_y_m": gps_noise[:, 1],
    "lidar_error_x_m": lidar_noise[:, 0],
    "lidar_error_y_m": lidar_noise[:, 1],
})

robot_df.to_csv(
    DATASET_ROOT / "robotics" / "robot_sensor_log.csv",
    index=False,
)

robot_df.head()


,time_s,true_x_m,true_y_m,true_velocity_mps,true_heading_rad,gps_x_m,gps_y_m,lidar_x_m,lidar_y_m,velocity_measurement_mps,heading_measurement_rad,gps_error_x_m,gps_error_y_m,lidar_error_x_m,lidar_error_y_m
0,0.0,0.000000,0.000000,1.280000,0.040000,0.389171,-0.116192,-0.186955,0.000243,1.368620,0.057620,0.389171,-0.116192,-0.186955,0.000243
1,0.1,0.127898,0.005119,1.284499,0.041800,-0.127170,0.305739,0.147994,0.067447,1.211959,0.055796,-0.255068,0.300621,0.020097,0.062329
2,0.2,0.256235,0.010486,1.288994,0.043599,0.283851,0.160250,0.495614,0.418710,1.165425,0.037359,0.027616,0.149763,0.239378,0.408224
3,0.3,0.385012,0.016104,1.293484,0.045397,-0.059789,0.137527,0.093869,-0.082504,1.349801,0.062646,-0.444801,0.121423,-0.291144,-0.098608
4,0.4,0.514227,0.021974,1.297968,0.047194,0.915233,0.474460,0.072914,0.398295,1.263833,0.063715,0.401005,0.452486,-0.441314,0.376320


## 5. Robot Dataset with Missing Measurements

We remove approximately:

- 15% of GPS measurements,
- 10% of LiDAR measurements,
- 8% of velocity measurements.

This dataset will be used later for learning with missing data.


In [8]:
robot_missing_df = robot_df.copy()

rng = np.random.default_rng(46)

gps_missing = rng.choice(
    n_steps,
    size=int(0.15 * n_steps),
    replace=False,
)

lidar_missing = rng.choice(
    n_steps,
    size=int(0.10 * n_steps),
    replace=False,
)

velocity_missing = rng.choice(
    n_steps,
    size=int(0.08 * n_steps),
    replace=False,
)

robot_missing_df.loc[
    gps_missing,
    ["gps_x_m", "gps_y_m"],
] = np.nan

robot_missing_df.loc[
    lidar_missing,
    ["lidar_x_m", "lidar_y_m"],
] = np.nan

robot_missing_df.loc[
    velocity_missing,
    ["velocity_measurement_mps"],
] = np.nan

robot_missing_df.to_csv(
    DATASET_ROOT / "robotics" / "robot_sensor_missing.csv",
    index=False,
)

robot_missing_df.head(15)


,time_s,true_x_m,true_y_m,true_velocity_mps,true_heading_rad,gps_x_m,gps_y_m,lidar_x_m,lidar_y_m,velocity_measurement_mps,heading_measurement_rad,gps_error_x_m,gps_error_y_m,lidar_error_x_m,lidar_error_y_m
0,0.0,0.000000,0.000000,1.280000,0.040000,0.389171,-0.116192,-0.186955,0.000243,1.368620,0.057620,0.389171,-0.116192,-0.186955,0.000243
1,0.1,0.127898,0.005119,1.284499,0.041800,-0.127170,0.305739,0.147994,0.067447,1.211959,0.055796,-0.255068,0.300621,0.020097,0.062329
2,0.2,0.256235,0.010486,1.288994,0.043599,NaN,NaN,0.495614,0.418710,1.165425,0.037359,0.027616,0.149763,0.239378,0.408224
3,0.3,0.385012,0.016104,1.293484,0.045397,-0.059789,0.137527,0.093869,-0.082504,1.349801,0.062646,-0.444801,0.121423,-0.291144,-0.098608
4,0.4,0.514227,0.021974,1.297968,0.047194,0.915233,0.474460,0.072914,0.398295,1.263833,0.063715,0.401005,0.452486,-0.441314,0.376320
5,0.5,0.643880,0.028098,1.302445,0.048990,NaN,NaN,1.027546,0.204626,1.397424,0.060115,0.137654,-0.237457,0.383666,0.176528
6,0.6,0.773968,0.034476,1.306912,0.050784,1.202485,0.142823,0.804758,-0.066054,NaN,0.116449,0.428517,0.108348,0.030791,-0.100530
7,0.7,0.904491,0.041110,1.311368,0.052576,1.771763,-0.254332,1.413563,0.503949,1.281716,0.050594,0.867273,-0.295442,0.509072,0.462839
8,0.8,1.035446,0.048002,1.315812,0.054366,0.835251,0.341739,1.466335,0.071156,1.299452,0.049765,-0.200195,0.293737,0.430889,0.023154
9,0.9,1.166833,0.055152,1.320242,0.056154,1.145056,0.424982,1.238625,0.444266,1.376503,0.040345,-0.021776,0.369831,0.071792,0.389115


## 6. Dataset Summary

| Dataset | Main use |
|---|---|
| `coin_toss.csv` | Bernoulli MLE and Bayesian learning |
| `dice_rolls.csv` | Categorical parameter estimation |
| `gaussian_samples.csv` | Gaussian mean/covariance estimation |
| `robot_sensor_log.csv` | Robotics parameter-learning experiments |
| `robot_sensor_missing.csv` | Missing-data learning |
